In [1]:
import pandas as pd
from statsmodels.tsa.ardl import ardl_select_order

df = pd.read_csv(
    "../data/processed/passthrough_dataset.csv",
    parse_dates=["date"]
).set_index("date")

df = df[["ron97", "wti", "usdmyr"]]
df = df.asfreq("ME")

df.head()


,ron97,wti,usdmyr
date,,,
2020-01-31,2.5725,57.519048,4.080110
2020-02-29,2.3780,50.542632,4.159900
2020-03-31,1.9275,29.207727,4.298409
2020-04-30,1.5625,16.547619,4.352486
2020-05-31,1.6240,28.562500,4.340906


In [2]:
sel_wti = ardl_select_order(
    endog=df["ron97"],
    exog=df[["wti", "usdmyr"]],
    maxlag=4,
    maxorder=4,
    ic="aic",
    trend="c"
)

print(sel_wti.model)


In [3]:
ardl_r97_wti = sel_wti.model.fit()
print(ardl_r97_wti.summary())


                              ARDL Model Results                              
Dep. Variable:                  ron97   No. Observations:                   71
Model:                     ARDL(2, 1)   Log Likelihood                  70.369
Method:               Conditional MLE   S.D. of innovations              0.087
Date:                Thu, 08 Jan 2026   AIC                           -128.738
Time:                        09:18:21   BIC                           -115.333
Sample:                    03-31-2020   HQIC                          -123.420
                         - 11-30-2025                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0548      0.053      1.035      0.304      -0.051       0.161
ron97.L1       1.1417      0.114     10.040      0.000       0.915       1.369
ron97.L2      -0.2950      0.083     -3.575      0.0

In [4]:
params = ardl_r97_wti.params

phi_sum = sum(v for k,v in params.items() if "ron97.L" in k)
den = 1 - phi_sum

b_sum = sum(v for k,v in params.items() if "wti.L" in k)

theta_wti = b_sum / den

print("Long-run WTI effect on RON97:", theta_wti)


Long-run WTI effect on RON97: 0.03954733110871838


In [5]:
import numpy as np
import statsmodels.api as sm
from scipy import stats
from statsmodels.tsa.tsatools import lagmat
from statsmodels.stats.diagnostic import het_breuschpagan


In [6]:
res = ardl_r97_wti

def bg_test_ardl_aligned(res, nlags=2):
    X0_full = np.asarray(res.model.data.exog)
    u = np.asarray(res.resid)

    n_u = len(u)
    if n_u <= nlags + 5:
        raise ValueError(f"Too few residual observations ({n_u}) for nlags={nlags}.")

    # align exog to residual sample length
    X0 = X0_full[-n_u:, :]

    # auxiliary regression: u_t on X + lagged u
    U_lags = lagmat(u, maxlag=nlags, trim="both")
    y = u[nlags:]
    X = np.column_stack([X0[nlags:, :], U_lags])

    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Non-finite values (NaN/inf) in BG auxiliary regression inputs.")

    aux_res = sm.OLS(y, X).fit()

    n = aux_res.nobs
    lm = n * aux_res.rsquared
    lm_p = stats.chi2.sf(lm, nlags)

    k = X.shape[1]
    R = np.zeros((nlags, k))
    R[:, -nlags:] = np.eye(nlags)
    f_test = aux_res.f_test(R)

    return {
        "LM stat": float(lm),
        "LM p-value": float(lm_p),
        "F stat": float(f_test.fvalue),
        "F p-value": float(f_test.pvalue),
        "nobs_aux": int(n),
        "resid_len": int(n_u),
        "exog_rows_full": int(X0_full.shape[0]),
    }

bg_out = bg_test_ardl_aligned(res, nlags=2)
bg_out


{'LM stat': 0.3203337483212312,
 'LM p-value': 0.8520016000508617,
 'F stat': 0.13411923273862356,
 'F p-value': 0.8747348060982977,
 'nobs_aux': 67,
 'resid_len': 69,
 'exog_rows_full': 71}

In [7]:
def bp_test_ardl(res):
    u = np.asarray(res.resid)
    X0_full = np.asarray(res.model.data.exog)
    X0 = X0_full[-len(u):, :]  # align
    X0 = sm.add_constant(X0, has_constant="add")  # force constant

    bp = het_breuschpagan(u, X0)

    return {
        "LM stat": float(bp[0]),
        "LM p-value": float(bp[1]),
        "F stat": float(bp[2]),
        "F p-value": float(bp[3]),
        "k_exog": int(X0.shape[1]),
    }

bp_out = bp_test_ardl(res)
bp_out


{'LM stat': 16.437220663541588,
 'LM p-value': 0.00026958944459971494,
 'F stat': 10.319627096290837,
 'F p-value': 0.00012599827610117846,
 'k_exog': 3}

In [8]:
import joblib
joblib.dump(res, "../data/joblib/ardl_ron97_wti.joblib")

['../data/joblib/ardl_ron97_wti.joblib']